# SET OS Camera Coach — ML-004 production heads training (human-gold gated)

Research/training orchestration template. Order: **preflight → mount Drive → verify data hashes → smoke → full train → eval → export → receipts**.

Boundary (read before running):

- The bundle is fail-closed: without adjudicated `human_gold=true` labels only `dry_run`/`smoke` are permitted. **Silver data is never gold.**
- The locked test split is **forbidden** in this notebook until ML-008 (single locked evaluation). Eval here runs on validation only.
- Every checkpoint, metric and receipt is written to Drive so a disconnected runtime loses nothing; training resumes from the latest checkpoint.
- The model proposes verdict/issue/action/target/delta; the app still verifies everything (owner decision 2026-09-22).

In [ ]:
# 1. Preflight (fail-closed). Refuses to continue on any drift.
import json
import subprocess
import sys
from pathlib import Path

BUNDLE_ROOT = Path('/content/camera-coach-ml004-colab-bundle')  # extracted tar.gz root
if not BUNDLE_ROOT.is_dir():
    raise RuntimeError(f'extract the bundle tar.gz first; missing {BUNDLE_ROOT}')
MODE = 'dry_run'  # dry_run | smoke | final_fit (final_fit requires human gold; see cell 4)
proc = subprocess.run(
    [sys.executable, str(BUNDLE_ROOT / 'check_environment.py'), '--mode', MODE],
    capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-2000:])
    raise RuntimeError(f'preflight failed (exit {proc.returncode}); fix the environment before continuing')
sys.path.insert(0, str(BUNDLE_ROOT / 'bundle'))  # ml.* namespace from the hashed bundle closure

In [ ]:
# 2. Mount Google Drive (persistent storage; media never enters the bundle).
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
RUN_ROOT = DRIVE_ROOT / 'runs' / 'ml004'  # checkpoints, receipts.jsonl, exports
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f'run root: {RUN_ROOT}')

In [ ]:
# 3. Verify data hashes at Drive paths (the bundle pins content_sha256 per file).
# Full pass over all pinned rows; a Colab runtime needs a few minutes for ~12.5k files.
proc = subprocess.run(
    [sys.executable, str(BUNDLE_ROOT / 'check_environment.py'), '--mode', MODE, '--full'],
    capture_output=True, text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    raise RuntimeError('data hash verification failed; re-upload the changed files, never patch hashes')
# Speed tip (bundle README): copy verified frames to the local /content SSD once,
# then read from the local copy during training. Drive is persistent, not fast.

In [ ]:
# 4. Human-gold slot — TODO(OWNER): stub until the markup lands.
#
# When pilot-100 labels (and later DATA-013 episodes) are adjudicated (DATA-010/011)
# and uploaded to Drive:
#   1. set MODE = 'final_fit' in cell 1 and re-run the preflight; the gate then
#      validates every envelope against camera_release_label_v1 and refuses unless
#      every row is human_gold=true;
#   2. build the record loader from release-label envelopes:
#      from ml.camera_coach.production_heads import supervision_from_release_label
#      (each envelope['release_label'] -> per-head targets/masks; a missing label
#      stays masked, it never becomes a zero target);
#   3. freeze splits per DATA-012 manifests (source-family grouped, leakage-safe)
#      before the first optimizer step.
slot = {}
for line in (BUNDLE_ROOT / 'data_manifest.jsonl').read_text(encoding='utf-8').splitlines():
    if line.strip() and 'human_labels_dir' in line:
        slot = json.loads(line)
        break
print('human-gold slot status:', slot.get('status'))
print('expected:', json.dumps(slot.get('expected', {}), indent=2)[:800])
if MODE == 'final_fit' and slot.get('status') == 'awaiting_owner':
    raise RuntimeError('final_fit is blocked: the human-gold slot is still awaiting_owner')

In [ ]:
# 5. Smoke: synthetic mini-batch through the pinned production heads —
# layout, masked losses, backward reach. Runnable today, no data needed.
# NOTE: an issues list, when present, must cover all 15 heads (present/absent/
# not_observed); a partially listed verdict is rejected fail-closed.
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from ml.camera_coach.production_heads import (  # noqa: E402
    ProductionHeadsScorer,
    ProductionHeadsSpec,
    compute_production_heads_loss,
    supervision_from_release_label,
)

spec = ProductionHeadsSpec.load()
print('head widths:', dict(spec.head_widths))
scorer = ProductionHeadsScorer(spec).to(DEVICE)
records = [
    {  # corrective verdict: all 15 issue heads covered (present/absent/not_observed)
        'verdict': 'needs_fix',
        'issues': [
            {'issue_id': issue_id,
             'status': ('present' if n == 0 else 'not_observed' if n == 2 else 'absent')}
            for n, issue_id in enumerate(spec.issue_ids)
        ],
        'action': None,
        'risk': None,
        'abstention': None,
    },
    {  # good frame: every issue an explicit negative, adjudicated abstention none
        'verdict': 'good',
        'issues': [{'issue_id': issue_id, 'status': 'absent'} for issue_id in spec.issue_ids],
        'action': None,
        'risk': None,
        'abstention': {'status': 'none'},
    },
    {  # fully unadjudicated row: every mask must be 0
        'verdict': None,
        'issues': None,
        'action': None,
        'risk': None,
        'abstention': None,
    },
]
supervisions = [supervision_from_release_label(record, spec) for record in records]
targets, masks = supervisions[0].collate(supervisions[1:])
features = torch.randn(len(records), 32, device=DEVICE)
outputs = {head: value.to(DEVICE) for head, value in scorer(features).items()}
loss = compute_production_heads_loss(outputs, targets, masks, spec)
loss.total.backward()
assert torch.isfinite(loss.total), 'smoke loss must stay finite'
assert float(masks['good_frame_probability'][2]) == 0.0, 'unadjudicated row must be fully masked'
for head in ('good_frame_probability', 'issue_logits', 'action_utility_logits',
             'continuous_target_deltas', 'risk_probability', 'abstention_probability'):
    grad = scorer.heads[head].weight.grad
    assert torch.isfinite(grad).all(), f'{head} gradient must stay finite'
print('smoke loss:', float(loss.total.detach()))
print('per-head:', {head: round(float(value.detach()), 6) for head, value in loss.per_head.items()})
print('smoke PASS: shapes, masked losses and backward all finite')


In [ ]:
# 6. Full train — TODO(OWNER): the entrypoint lands with ML-005 (frozen DATA-016).
# Skeleton (runs only after the human-gold gate admits final_fit):
#
#   config = json.loads((BUNDLE_ROOT / 'config.json').read_text())
#   torch.manual_seed(config['seed'])
#   model = build_backbone(config)             # ML-005 selects the final architecture
#   optimizer = torch.optim.AdamW(model.parameters(),
#                                 lr=config['training']['learning_rate'],
#                                 weight_decay=config['training']['weight_decay'])
#   ckpt_dir = Path(config['resume']['checkpoint_dir'])
#   model, optimizer, start_step = resume_latest(ckpt_dir)   # resume from Drive
#   for step, (images, targets, masks) in enumerate(train_loader, start=start_step):
#       loss = compute_production_heads_loss(model(images), targets, masks, spec,
#                                            weights=config['masked_losses']['loss_weights'])
#       loss.total.backward(); optimizer.step(); optimizer.zero_grad()
#       if step % config['resume']['checkpoint_every_steps'] == 0:
#           write_checkpoint(ckpt_dir, model, optimizer, step)  # atomic to Drive, keep_last_k
#
# Rules: early stopping on validation only; the locked test is never mounted here;
# every checkpoint write flushes to Drive before training continues.
print('full train: TODO(owner) — requires the ML-005 entrypoint + adjudicated human-gold data')

In [ ]:
# 7. Eval — per-head metrics on VALIDATION only (ML-002 anchors), never the locked test.
#
# Per head, never one averaged accuracy:
#   good_frame_probability  : AUROC / ECE / Brier
#   issue_logits (15)       : per-head AUROC/ECE + good-frame false-positive rate
#   action_utility_logits   : masked top-1 per family; wrong-direction separate
#   continuous_target_deltas: masked MAE/RMSE + within-verifier-deadband rate
#   risk / abstention       : AUROC/ECE + risk-coverage curve
# Safety failures are reported as their own counts and are never averaged in.
print('eval: per-head metrics scaffold activates with the first ML-005 checkpoint')
print('locked test: FORBIDDEN in this notebook until ML-008')

In [ ]:
# 8. Export FP16 Core ML (ML-012 preparation) — TODO(OWNER) until admission inputs exist.
#
#   import coremltools as ct
#   model.eval()
#   traced = torch.jit.trace(model, example_inputs)
#   mlmodel = ct.convert(traced, compute_precision=ct.precision.FLOAT16,
#                        compute_units=ct.ComputeUnit.CPU_AND_NE)
#   # canonical I/O names/shapes per the pinned contract; calibration metadata is a
#   # separate artifact (CAM-003/ML-007), never baked into the weights.
#   mlmodel.save(str(RUN_ROOT / 'candidate.mlpackage'))
# The real export is ML-012 work and requires ML-007/008/009 inputs.
print('coreml export: TODO(owner) — ML-012, after ML-007/008/009')

In [ ]:
# 9. Receipts: append one JSONL line per stage to Drive (hashes, config, metrics).
import datetime as dt
import hashlib


def _sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


receipts_path = RUN_ROOT / 'receipts.jsonl'
entry = {
    'utc': dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat(),
    'mode': MODE,
    'bundle_receipt_sha256': _sha256(BUNDLE_ROOT / 'bundle-receipt.json'),
    'config_sha256': _sha256(BUNDLE_ROOT / 'config.json'),
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'human_gold_slot': slot.get('status'),
    # 'checkpoint_sha256': ..., 'metrics_sha256': ...  # filled at train/eval stages
}
with receipts_path.open('a', encoding='utf-8') as stream:
    stream.write(json.dumps(entry, ensure_ascii=False) + '\n')
print(f'receipt appended: {receipts_path}')
print(receipts_path.read_text())

## Boundary

- This notebook never touches the locked test split (ML-008 owns the single evaluation).
- Silver pairs and research-package frames are context/negative-mining material only; a final fit is admissible exclusively on adjudicated human-gold labels (DATA-010…016).
- No export from this notebook is production-admitted; Core ML admission is ML-012…ML-018.
- Do not edit bundle files in place: a hash drift is a rebuild (`build_bundle.py`), not a patch.